# All four research questions, up close

This notebook shows what the one evaluation cycle (`evaluation/run.py`) produced, question by question. Everything is read from files on disk, nothing is computed fresh, and every section works on a partial run: sections whose files do not exist yet say so instead of crashing. Re-execute the cells any time to refresh.

Where the data lives:
- `evaluation/out/predictions/v1|v2/` one file per record x variant x pipeline
- `evaluation/rq1..rq4/out/` each question's `results.csv` and `scores.json`

In [18]:
# The repository root is the first parent directory holding docker-compose.yml
import csv
import json
from pathlib import Path

REPO = Path.cwd().resolve()
while not REPO.joinpath("docker-compose.yml").exists():
  REPO = REPO.parent

STORE = REPO.joinpath("evaluation", "out", "predictions")
RQ_OUT = {rq: REPO.joinpath("evaluation", rq, "out") for rq in ("rq1", "rq2", "rq3", "rq4")}
DATASETS = REPO.joinpath("dataprep", "out", "testing")


def read_scores(rq):
  path = RQ_OUT[rq].joinpath("scores.json")
  if not path.exists():
    print(f"no {rq} scores yet; run evaluation/run.py (the scorers run at the end)")
    return None
  return json.loads(path.read_text())


def read_results(rq):
  path = RQ_OUT[rq].joinpath("results.csv")
  if not path.exists():
    return []
  return list(csv.DictReader(open(path)))


print("store:", STORE)

store: /home/amed/uni/insight/evaluation/out/predictions


## 0 Run progress

One prediction file per cell. A full run is 1,611 cells: 861 text (three batches x three pipelines), 600 audio-batch cells (50 calls x 4 variants x 3 pipelines), and 150 schema-v2 cells for rq4. Status meaning:
- `complete` scored
- `failed` core failed the record (for v2 p1 cells this is the designed refusal)
- `timeout` / `upload_failed` infrastructure problems, never counted as wrong answers

In [19]:
# Count the stored cells per schema x batch x variant x pipeline x status
counts = {}
for schema_dir in ("v1", "v2"):
  for path in sorted(STORE.joinpath(schema_dir).glob("*.json")) if STORE.joinpath(schema_dir).exists() else []:
    batch, rest = path.stem.split("__", 1)
    record_id, variant, pipeline = rest.rsplit("__", 2)
    status = json.loads(path.read_text())["status"]
    key = (schema_dir, batch, variant, pipeline, status)
    counts[key] = counts.get(key, 0) + 1

total = sum(counts.values())
print(f"{total}/1611 cells stored\n")
print(f"{'schema':<7} {'batch':<12} {'variant':<8} {'pipeline':<9} {'status':<14} count")
for (schema_dir, batch, variant, pipeline, status), count in sorted(counts.items()):
  print(f"{schema_dir:<7} {batch:<12} {variant:<8} {pipeline:<9} {status:<14} {count}")
if not counts:
  print("nothing yet, run evaluation/run.py first")

1611/1611 cells stored

schema  batch        variant  pipeline  status         count
v1      abcd-text    text     p1        complete       186
v1      abcd-text    text     p2        complete       186
v1      abcd-text    text     p3        complete       186
v1      hvb-audio    asr      p1        complete       50
v1      hvb-audio    asr      p2        complete       50
v1      hvb-audio    asr      p3        complete       50
v1      hvb-audio    mono     p1        complete       50
v1      hvb-audio    mono     p2        complete       50
v1      hvb-audio    mono     p3        complete       50
v1      hvb-audio    oracle   p1        complete       50
v1      hvb-audio    oracle   p2        complete       50
v1      hvb-audio    oracle   p3        complete       50
v1      hvb-audio    stereo   p1        complete       50
v1      hvb-audio    stereo   p2        complete       50
v1      hvb-audio    stereo   p3        complete       50
v1      maia-text    text     p1        co

## 1 RQ1: which pipeline extracts the fields most accurately

- `accuracy` share of scored records where the prediction equals gold
- `macro_f1` per-class f1 averaged over the classes that appear in gold, so a pipeline can not look good by only getting the most common class right
- `not_complete` failed or timed out cells, excluded from both

Each batch scores only the fields it has gold for.

In [20]:
scores = read_scores("rq1")
if scores:
  print(f"{'batch':<12} {'pipeline':<9} {'field':<18} {'accuracy':<9} {'macro_f1':<9} {'scored':<7} not_complete")
  for batch, pipelines in scores.items():
    for pipeline, per_field in pipelines.items():
      for field, s in per_field.items():
        acc = f"{s['accuracy']:.2f}" if s["accuracy"] is not None else "-"
        f1 = f"{s['macro_f1']:.2f}" if s["macro_f1"] is not None else "-"
        print(f"{batch:<12} {pipeline:<9} {field:<18} {acc:<9} {f1:<9} {s['scored']:<7} {s['not_complete']}")

batch        pipeline  field              accuracy  macro_f1  scored  not_complete
abcd-text    p1        agent_action       0.84      0.80      186     0
abcd-text    p1        intent             0.94      0.88      174     0
abcd-text    p1        issue_type         0.91      0.87      186     0
abcd-text    p2        agent_action       0.26      0.35      186     0
abcd-text    p2        intent             0.44      0.30      174     0
abcd-text    p2        issue_type         0.34      0.33      186     0
abcd-text    p3        agent_action       0.53      0.45      186     0
abcd-text    p3        intent             0.74      0.57      174     0
abcd-text    p3        issue_type         0.61      0.51      186     0
hvb-audio    p1        intent             0.77      0.67      26      0
hvb-audio    p2        intent             0.23      0.33      26      0
hvb-audio    p3        intent             0.96      0.96      26      0
maia-text    p1        resolution_status  0.77      0

## 1b One record up close

The conversation as the pipelines saw it, then gold next to each pipeline's answer. Change `sample_index` to look at another record.

In [21]:
sample_index = 0
v1 = STORE.joinpath("v1")
samples = sorted(v1.glob("*__text__p2.json")) + sorted(v1.glob("*__oracle__p2.json")) if v1.exists() else []
if not samples:
  print("no p2 cells stored yet")
else:
  sample = samples[sample_index]
  batch, rest = sample.stem.split("__", 1)
  record_id, variant, _ = rest.rsplit("__", 2)

  record = json.loads(DATASETS.joinpath(batch, "records", f"{record_id}.json").read_text())
  print(f"record {record_id} from {batch}\n")
  for turn in record["turns"][:8]:
    print(f"{turn['speaker']:>9}: {turn['text']}")
  if len(record["turns"]) > 8:
    print(f"... {len(record['turns']) - 8} more turns")

  gold = json.loads(DATASETS.joinpath(batch, "gold.json").read_text())[record_id]
  predictions = {}
  for pipeline in ("p1", "p2", "p3"):
    path = v1.joinpath(f"{batch}__{record_id}__{variant}__{pipeline}.json")
    predictions[pipeline] = json.loads(path.read_text()) if path.exists() else {"fields": {}, "status": "not run"}

  fields = [f for f in gold if not f.endswith("_source") and f != "speakers"]
  print(f"\n{'field':<18} {'gold':<20} {'p1':<20} {'p2':<20} {'p3':<20}")
  for field in fields:
    row = f"{field:<18} {str(gold[field]):<20}"
    for pipeline in ("p1", "p2", "p3"):
      row += f"{str(predictions[pipeline]['fields'].get(field, '')):<20}"
    print(row)

record abcd-10012 from abcd-text

    agent: Hello, how may I help you today?
 customer: Hello. I need to check on a recent refund for a return i did.
    agent: I see, let's see what we can found out.  One moment.
    agent: May I have your full name or account ID?
 customer: yes it's Rodriguez Domingo and i don't know my account id.
    agent: That is fine, I will need to validate your purchase.  May I have your email address and order ID?
    agent: Oh and your username please as well.
 customer: sure my email is rd449161@email.com and my username is rd449161
... 10 more turns

field              gold                 p1                   p2                   p3                  
intent             refund_request      refund_request      unknown             refund_request      
issue_type         product_defect      product_defect      unknown             none                
agent_action       none                none                provide_information provide_information 


## 2 RQ2: does grounding reduce unsupported claims

Only p2 and p3 are compared, p1 can not cite. A claim-bearing value (not unknown, not none) is `supported` when at least one citation survived the grounding filter; `uncited` when the model cited nothing; `invalid-cited` when everything it cited was dropped. `ucr` = unsupported claims / claims. `validity` = share of fields whose first answer was already a legal schema value.

The pre-committed criteria: UCR(p2) < UCR(p3), validity at least 0.95 for both.

In [22]:
scores = read_scores("rq2")
if scores:
  print(f"{'pipeline':<9} {'scope':<12} {'claims':<7} {'supported':<10} {'uncited':<8} {'invalid':<8} {'ucr':<7} validity")
  for pipeline in ("p2", "p3"):
    for scope, t in scores[pipeline].items():
      ucr = f"{t['ucr']:.3f}" if t["ucr"] is not None else "-"
      validity = f"{t['validity']:.3f}" if t["validity"] is not None else "-"
      print(f"{pipeline:<9} {scope:<12} {t['claim_bearing']:<7} {t['supported']:<10} "
            f"{t['uncited']:<8} {t['invalid_cited']:<8} {ucr:<7} {validity}")
  print("\ncriteria:")
  print(json.dumps(scores["criteria"], indent=1))

pipeline  scope        claims  supported  uncited  invalid  ucr     validity
p2        abcd-text    419     419        0        0        0.000   1.000
p2        hvb-audio    79      79         0        0        0.000   1.000
p2        maia-text    348     348        0        0        0.000   1.000
p2        overall      846     846        0        0        0.000   1.000
p3        abcd-text    668     667        0        1        0.001   1.000
p3        hvb-audio    154     154        0        0        0.000   1.000
p3        maia-text    405     404        0        1        0.002   1.000
p3        overall      1227    1225       0        2        0.002   1.000

criteria:
{
 "ucr_p2_lt_p3": {
  "p2": 0.0,
  "p3": 0.0016299918500407497,
  "met": true
 },
 "validity_ge_95": {
  "p2": {
   "validity": 1.0,
   "met": true
  },
  "p3": {
   "validity": 1.0,
   "met": true
  }
 }
}


## 2b An unsupported claim, concretely

The first stored claim that has no surviving citation: the pipeline asserted a value and could not point at a line that backs it.

In [23]:
rows = read_results("rq2")
unsupported = [r for r in rows if r["claim_bearing"] == "1" and r["supported"] == "0"]
if not unsupported:
  print("no rq2 results yet, or every claim was supported")
else:
  example = unsupported[0]
  print(f"{example['pipeline']} on {example['record']} ({example['batch']}):")
  print(f"  {example['field']} = {example['value']!r}")
  print(f"  citations: {example['citations']}, dropped by the filter: {example['dropped']}")
  print(f"\n{len(unsupported)} unsupported claims in total")

p3 on abcd-3034 (abcd-text):
  intent = 'information_request'
  citations: 0, dropped by the filter: 2

2 unsupported claims in total


## 3 RQ3: how much does the audio path cost

The same 50 calls in four variants. `oracle` (the human transcript) is the reference; `asr` is the corpus machine transcript; `mono` and `stereo` go through whisper, mono also through diarisation.
- `wer` word error rate of the produced transcript against the oracle turns
- `role` speaker assignment accuracy (mono/stereo only, overlap matched)
- `to_wrong` answers that were right on oracle but wrong on this variant, same call, same pipeline; `to_right` is the reverse

In [24]:
scores = read_scores("rq3")
if scores:
  print(f"{'variant':<8} {'pipeline':<9} {'accuracy':<9} {'wer':<7} {'role':<7} {'to_wrong':<9} {'to_right':<9} not_complete")
  for variant in ("oracle", "asr", "mono", "stereo"):
    for pipeline, s in scores.get(variant, {}).items():
      acc = f"{s['accuracy']:.2f}" if s["accuracy"] is not None else "-"
      wer = f"{s['mean_wer']:.3f}" if s["mean_wer"] is not None else "-"
      role = f"{s['mean_role_accuracy']:.3f}" if s["mean_role_accuracy"] is not None else "-"
      print(f"{variant:<8} {pipeline:<9} {acc:<9} {wer:<7} {role:<7} "
            f"{s['to_wrong']:<9} {s['to_right']:<9} {s['not_complete']}")

variant  pipeline  accuracy  wer     role    to_wrong  to_right  not_complete
oracle   p1        0.77      -       -       0         0         0
oracle   p2        0.23      -       -       0         0         0
oracle   p3        0.96      -       -       0         0         0
asr      p1        0.77      0.098   -       0         0         0
asr      p2        0.23      0.098   -       0         0         0
asr      p3        0.85      0.098   -       3         0         0
mono     p1        0.77      0.219   0.808   0         0         0
mono     p2        0.31      0.219   0.808   0         2         0
mono     p3        0.92      0.219   0.808   2         1         0
stereo   p1        0.77      0.386   0.964   0         0         0
stereo   p2        0.27      0.383   0.964   1         2         0
stereo   p3        0.92      0.390   0.965   2         1         0


## 4 RQ4: can a field be added by configuration alone

Schema v2 = v1 plus `banking_task`, added as one json file, no retraining, no code change. Gold comes free from the corpus task labels. p1's cells fail on the schema claim by design: the supervised pipeline structurally can not follow a configuration-only change, and refusing beats guessing.

The pre-committed criteria: validity at least 0.95, and banking_task accuracy within 10 points of intent accuracy on the same calls.

In [25]:
scores = read_scores("rq4")
if scores:
  print(f"p1 refusal: {scores['p1']['failed']} of {scores['p1']['cells']} cells failed (designed)\n")
  print(f"{'pipeline':<9} {'banking_task':<13} {'intent':<8} {'validity':<9} supported_claims")
  for pipeline in ("p2", "p3"):
    s = scores[pipeline]
    banking = f"{s['banking_task_accuracy']:.2f}" if s["banking_task_accuracy"] is not None else "-"
    intent = f"{s['intent_accuracy']:.2f}" if s["intent_accuracy"] is not None else "-"
    validity = f"{s['validity']:.3f}" if s["validity"] is not None else "-"
    print(f"{pipeline:<9} {banking:<13} {intent:<8} {validity:<9} {s['supported_banking_claims']}")
  print("\ncriteria:")
  print(json.dumps(scores["criteria"], indent=1))

p1 refusal: 50 of 50 cells failed (designed)

pipeline  banking_task  intent   validity  supported_claims
p2        0.64          0.12     1.000     32
p3        0.46          0.46     1.000     23

criteria:
{
 "validity_ge_95": {
  "p2": {
   "validity": 1.0,
   "met": true
  },
  "p3": {
   "validity": 1.0,
   "met": true
  }
 },
 "banking_within_10_of_intent": {
  "p2": {
   "banking_task": 0.64,
   "intent": 0.11538461538461539,
   "met": true
  },
  "p3": {
   "banking_task": 0.46,
   "intent": 0.46153846153846156,
   "met": true
  }
 }
}


## 5 Reading guide

1. **Section 0** is the run monitor: how far the collection is and whether failures piled up. Retry infrastructure failures by deleting their files (keep the v2 p1 `failed` ones, that is the rq4 result) and rerunning `evaluation/run.py`.
2. **Section 1** answers rq1: per field, which pipeline is most accurate.
3. **Section 2** answers rq2: whether grounding buys evidential support, and whether the pre-committed criteria hold.
4. **Section 3** answers rq3: extraction degradation as transcription quality drops, with the transcription and speaker errors measured separately.
5. **Section 4** answers rq4: extension by configuration, with p1's refusal as the architectural half of the answer.

The same numbers, machine readable, live in `evaluation/rq1..rq4/out/`.